# Delta Lake Assignment: Incremental Data Processing (SCD1 and SCD2)

Objective: perform incremental data processing using Delta Lake on customer data derived from the Superstore dataset.

Steps covered: load data into a Delta table, clean it, simulate an incremental batch of new and updated records, apply a MERGE operation (both SCD Type 1 and SCD Type 2), validate the results, and display the final dataset.

## Environment setup

Point Spark at the local Hadoop winutils build (Windows requires this for Spark's file system layer to work).

In [1]:
import os
os.environ["HADOOP_HOME"] = "C:\\hadoop"
os.environ["PATH"] += os.pathsep + "C:\\hadoop\\bin"

## Start Spark with Delta Lake support

In [2]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder.appName("DeltaSCDAssignment")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)

Spark version: 3.5.3


## Step 0: Load the source data

Load the raw Superstore dataset in pandas to build the customer master table from.

In [3]:
import pandas as pd

df = pd.read_csv("../data/superstore.csv", encoding="latin1")
print(df.shape)
print(df.columns.tolist())

(9994, 21)
['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']


Deduplicate by Customer ID to get one row per customer.

In [4]:
cust_cols = ['Customer ID','Customer Name','Segment','Country','City','State','Postal Code','Region']
master = df[cust_cols].drop_duplicates(subset='Customer ID').reset_index(drop=True)
print(master.shape)
master.head()

(793, 8)


,Customer ID,Customer Name,Segment,Country,City,State,Postal Code,Region
0,CG-12520,Claire Gute,Consumer,United States,Henderson,Kentucky,42420,South
1,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,California,90036,West
2,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,Florida,33311,South
3,BH-11710,Brosina Hoffman,Consumer,United States,Los Angeles,California,90032,West
4,AA-10480,Andrew Allen,Consumer,United States,Concord,North Carolina,28027,South


Inject a few nulls and duplicate rows on purpose, so the cleaning step has real data quality issues to fix.

In [5]:
import numpy as np

np.random.seed(42)

# rename to snake_case for convenience
master = master.rename(columns={
    'Customer ID':'customer_id','Customer Name':'customer_name','Segment':'segment',
    'Country':'country','City':'city','State':'state','Postal Code':'postal_code','Region':'region'
})

# introduce a few nulls
null_idx = np.random.choice(master.index, size=8, replace=False)
master.loc[null_idx[:4], 'segment'] = np.nan
master.loc[null_idx[4:], 'postal_code'] = np.nan

# introduce a few exact duplicate rows
dupes = master.sample(5, random_state=1)
master = pd.concat([master, dupes], ignore_index=True)

print(master.shape)
master.isnull().sum()

(798, 8)


customer_id      0
customer_name    0
segment          4
country          0
city             0
state            0
postal_code      4
region           0
dtype: int64

Save as the master CSV, the source file for the Delta table.

In [6]:
master.to_csv("../data/customer_master.csv", index=False)

## Step 1: Load dataset into a Delta table

Read the CSV into Spark and confirm the data landed correctly before writing it as Delta.

In [7]:
master_path = "delta/customer_master"

master_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("../data/customer_master.csv")
)

print("Rows loaded:", master_df.count())
master_df.printSchema()
master_df.show(5, truncate=False)

Rows loaded: 798
root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- segment: string (nullable = true)
 |-- country: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- postal_code: double (nullable = true)
 |-- region: string (nullable = true)

+-----------+---------------+---------+-------------+---------------+--------------+-----------+------+
|customer_id|customer_name  |segment  |country      |city           |state         |postal_code|region|
+-----------+---------------+---------+-------------+---------------+--------------+-----------+------+
|CG-12520   |Claire Gute    |Consumer |United States|Henderson      |Kentucky      |42420.0    |South |
|DV-13045   |Darrin Van Huff|Corporate|United States|Los Angeles    |California    |90036.0    |West  |
|SO-20335   |Sean O'Donnell |Consumer |United States|Fort Lauderdale|Florida       |33311.0    |South |
|BH-11710   |Brosina Hoffman|Cons

Write it as an actual Delta table (not just a DataFrame).

In [8]:
master_df.write.format("delta").mode("overwrite").save(master_path)

spark.sql(f"CREATE TABLE IF NOT EXISTS customer_master USING DELTA LOCATION '{master_path}'")
print("Delta table created at:", master_path)

Delta table created at: delta/customer_master


Confirm the Delta log and parquet files were written to disk.

In [9]:
import os
for root, dirs, files in os.walk(master_path):
    for f in files:
        print(os.path.join(root, f))

delta/customer_master\.part-00000-2f2da409-8613-4e16-8e27-afb78cb48e42-c000.snappy.parquet.crc
delta/customer_master\.part-00000-33b73b34-024d-4264-94a3-19481433fa3a-c000.snappy.parquet.crc
delta/customer_master\.part-00000-41ee65bf-e195-4825-8d88-d93b47fc7a93-c000.snappy.parquet.crc
delta/customer_master\.part-00000-d1d627b4-b504-464f-8798-dd853eabd684-c000.snappy.parquet.crc
delta/customer_master\part-00000-2f2da409-8613-4e16-8e27-afb78cb48e42-c000.snappy.parquet
delta/customer_master\part-00000-33b73b34-024d-4264-94a3-19481433fa3a-c000.snappy.parquet
delta/customer_master\part-00000-41ee65bf-e195-4825-8d88-d93b47fc7a93-c000.snappy.parquet
delta/customer_master\part-00000-d1d627b4-b504-464f-8798-dd853eabd684-c000.snappy.parquet
delta/customer_master\_delta_log\.00000000000000000000.json.crc
delta/customer_master\_delta_log\.00000000000000000001.json.crc
delta/customer_master\_delta_log\.00000000000000000002.json.crc
delta/customer_master\_delta_log\.00000000000000000003.json.crc
delt

## Step 2: Basic cleaning

Check null counts and duplicate rows in the Delta table before cleaning.

In [10]:
import pyspark.sql.functions as F

raw = spark.read.format("delta").load(master_path)

print("Row count before cleaning:", raw.count())
print("Null counts per column:")
raw.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in raw.columns]).show()

exact_dupes = raw.count() - raw.dropDuplicates().count()
print("Exact duplicate rows:", exact_dupes)

Row count before cleaning: 798
Null counts per column:
+-----------+-------------+-------+-------+----+-----+-----------+------+
|customer_id|customer_name|segment|country|city|state|postal_code|region|
+-----------+-------------+-------+-------+----+-----+-----------+------+
|          0|            0|      4|      0|   0|    0|          4|     0|
+-----------+-------------+-------+-------+----+-----+-----------+------+

Exact duplicate rows: 5


Drop exact duplicates and rows missing segment or postal code, then overwrite the Delta table with the cleaned version.

In [11]:
cleaned = raw.dropDuplicates().dropna(subset=["segment", "postal_code"])

print("Row count after cleaning:", cleaned.count())

cleaned.write.format("delta").mode("overwrite").save(master_path)
print("customer_master Delta table updated with cleaned data.")

Row count after cleaning: 785
customer_master Delta table updated with cleaned data.


## Step 3: Create an incremental dataset

Simulate a new batch of data: 20 existing customers with updated segment/city values, plus 10 brand new customers.

In [12]:
# work from the cleaned master (785 unique, clean rows)
clean_master = master.dropna(subset=['segment','postal_code']).drop_duplicates(subset='customer_id').reset_index(drop=True)

# 20 existing customers get updated segment/city (simulates real changes)
updates = clean_master.sample(20, random_state=2).copy()
segments = ['Consumer','Corporate','Home Office']
updates['segment'] = [segments[i % 3] for i in range(len(updates))]
updates['city'] = updates['city'] + ' Heights'

# 10 brand new customers not in master at all
new_customers = pd.DataFrame({
    'customer_id': [f'NC-{20000+i}' for i in range(10)],
    'customer_name': [f'New Customer {i+1}' for i in range(10)],
    'segment': np.random.choice(segments, 10),
    'country': 'United States',
    'city': np.random.choice(['Austin','Denver','Seattle','Boston','Miami'], 10),
    'state': np.random.choice(['Texas','Colorado','Washington','Massachusetts','Florida'], 10),
    'postal_code': np.random.randint(10000, 99999, 10),
    'region': np.random.choice(['South','West','East','Central'], 10)
})

incremental = pd.concat([updates, new_customers], ignore_index=True)
incremental.to_csv("../data/customer_incremental.csv", index=False)
print(incremental.shape)
incremental.head()

(30, 8)


,customer_id,customer_name,segment,country,city,state,postal_code,region
0,ES-14080,Erin Smith,Consumer,United States,Melbourne Heights,Florida,32935.0,South
1,DH-13675,Duane Huffman,Corporate,United States,Quincy Heights,Massachusetts,2169.0,East
2,SW-20755,Steven Ward,Home Office,United States,New York City Heights,New York,10009.0,East
3,BF-10975,Barbara Fisher,Consumer,United States,Charlotte Heights,North Carolina,28205.0,South
4,DW-13480,Dianna Wilson,Corporate,United States,Lakeville Heights,Minnesota,55044.0,Central


Load the incremental batch into Spark.

In [13]:
incremental_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("../data/customer_incremental.csv")
)

print("Incremental rows:", incremental_df.count())
incremental_df.show(5, truncate=False)

Incremental rows: 30
+-----------+--------------+-----------+-------------+---------------------+--------------+-----------+-------+
|customer_id|customer_name |segment    |country      |city                 |state         |postal_code|region |
+-----------+--------------+-----------+-------------+---------------------+--------------+-----------+-------+
|ES-14080   |Erin Smith    |Consumer   |United States|Melbourne Heights    |Florida       |32935.0    |South  |
|DH-13675   |Duane Huffman |Corporate  |United States|Quincy Heights       |Massachusetts |2169.0     |East   |
|SW-20755   |Steven Ward   |Home Office|United States|New York City Heights|New York      |10009.0    |East   |
|BF-10975   |Barbara Fisher|Consumer   |United States|Charlotte Heights    |North Carolina|28205.0    |South  |
|DW-13480   |Dianna Wilson |Corporate  |United States|Lakeville Heights    |Minnesota     |55044.0    |Central|
+-----------+--------------+-----------+-------------+---------------------+-------

## Step 4: SCD Type 1 merge

Update matched rows in place, insert unmatched rows. No history is kept, this reflects only the latest state.

In [14]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark, master_path)

(
    delta_table.alias("target")
    .merge(
        incremental_df.alias("source"),
        "target.customer_id = source.customer_id"
    )
    .whenMatchedUpdate(set={
        "customer_name": "source.customer_name",
        "segment": "source.segment",
        "country": "source.country",
        "city": "source.city",
        "state": "source.state",
        "postal_code": "source.postal_code",
        "region": "source.region",
    })
    .whenNotMatchedInsertAll()
    .execute()
)

print("Merge complete.")
delta_table.toDF().orderBy("customer_id").show(10, truncate=False)

Merge complete.
+-----------+--------------------+-----------+-------------+-------------+--------------+-----------+-------+
|customer_id|customer_name       |segment    |country      |city         |state         |postal_code|region |
+-----------+--------------------+-----------+-------------+-------------+--------------+-----------+-------+
|AA-10315   |Alex Avila          |Consumer   |United States|Minneapolis  |Minnesota     |55407.0    |Central|
|AA-10375   |Allen Armold        |Consumer   |United States|Mesa         |Arizona       |85204.0    |West   |
|AA-10480   |Andrew Allen        |Consumer   |United States|Concord      |North Carolina|28027.0    |South  |
|AA-10645   |Anna Andreadi       |Consumer   |United States|Chester      |Pennsylvania  |19013.0    |East   |
|AB-10015   |Aaron Bergman       |Consumer   |United States|Seattle      |Washington    |98103.0    |West   |
|AB-10060   |Adam Bellavance     |Home Office|United States|New York City|New York      |10009.0    |Eas

## Step 5: Validate the SCD1 merge

Check row counts and confirm no duplicate customer_id values after merging.

In [15]:
scd1_df = delta_table.toDF()

print("Row count before merge (cleaned master):", 785)
print("Row count after merge:", scd1_df.count())

dupe_ids = scd1_df.groupBy("customer_id").count().filter("count > 1")
print("Duplicate customer_id count after merge:", dupe_ids.count())

# spot-check a few of the updated customers to confirm the 'Heights' change landed
updated_ids = [r["customer_id"] for r in incremental_df.limit(3).select("customer_id").collect()]
scd1_df.filter(F.col("customer_id").isin(updated_ids)).show(truncate=False)

Row count before merge (cleaned master): 785
Row count after merge: 795
Duplicate customer_id count after merge: 0
+-----------+-------------+-----------+-------------+---------------------+-------------+-----------+------+
|customer_id|customer_name|segment    |country      |city                 |state        |postal_code|region|
+-----------+-------------+-----------+-------------+---------------------+-------------+-----------+------+
|DH-13675   |Duane Huffman|Corporate  |United States|Quincy Heights       |Massachusetts|2169.0     |East  |
|ES-14080   |Erin Smith   |Consumer   |United States|Melbourne Heights    |Florida      |32935.0    |South |
|SW-20755   |Steven Ward  |Home Office|United States|New York City Heights|New York     |10009.0    |East  |
+-----------+-------------+-----------+-------------+---------------------+-------------+-----------+------+



## Step 6: Display final SCD1 result

In [16]:
print("=== Final customer_master Delta table ===")
scd1_df.orderBy("customer_id").show(10, truncate=False)

print("Total rows:", scd1_df.count())

=== Final customer_master Delta table ===
+-----------+--------------------+-----------+-------------+-------------+--------------+-----------+-------+
|customer_id|customer_name       |segment    |country      |city         |state         |postal_code|region |
+-----------+--------------------+-----------+-------------+-------------+--------------+-----------+-------+
|AA-10315   |Alex Avila          |Consumer   |United States|Minneapolis  |Minnesota     |55407.0    |Central|
|AA-10375   |Allen Armold        |Consumer   |United States|Mesa         |Arizona       |85204.0    |West   |
|AA-10480   |Andrew Allen        |Consumer   |United States|Concord      |North Carolina|28027.0    |South  |
|AA-10645   |Anna Andreadi       |Consumer   |United States|Chester      |Pennsylvania  |19013.0    |East   |
|AB-10015   |Aaron Bergman       |Consumer   |United States|Seattle      |Washington    |98103.0    |West   |
|AB-10060   |Adam Bellavance     |Home Office|United States|New York City|New 

Loaded the Superstore-derived customer master data (785 clean rows after removing 5 duplicate and 8 null-value rows) into a Delta table. Simulated an incremental batch of 30 records: 20 existing customers with updated segment and city values, and 10 brand new customers. Applied a MERGE INTO operation that updated matched customer records in place and inserted unmatched ones. Validated the result: row count increased from 785 to 795 (accounting for the 10 new customers), and a group-by check on customer_id confirmed zero duplicate keys after the merge. Numbers may change after re running script, so ignore if they don't match.

## SCD Type 2 merge

SCD1 overwrites old values with no record of what changed. SCD Type 2 keeps full history: expire the old row and insert a new current row instead of overwriting in place.

Seed a fresh Delta table from the cleaned master data, with effective_date, end_date, and is_current columns added.

In [17]:
scd2_path = "delta/customer_scd2"

scd2_seed = (
    spark.read.option("header", True).option("inferSchema", True)
    .csv("../data/customer_master.csv")
    .dropDuplicates()
    .dropna(subset=["segment", "postal_code"])
    .withColumn("effective_date", F.current_date())
    .withColumn("end_date", F.lit(None).cast("date"))
    .withColumn("is_current", F.lit(True))
)

scd2_seed.write.format("delta").mode("overwrite").save(scd2_path)
spark.sql(f"CREATE TABLE IF NOT EXISTS customer_scd2 USING DELTA LOCATION '{scd2_path}'")
print("SCD2 table seeded with", scd2_seed.count(), "rows")

SCD2 table seeded with 785 rows


Step A: expire the current row for any customer whose incoming data actually changed.

In [18]:
scd2_table = DeltaTable.forPath(spark, scd2_path)

staged = incremental_df.withColumn("effective_date", F.current_date()) \
                        .withColumn("end_date", F.lit(None).cast("date")) \
                        .withColumn("is_current", F.lit(True))

(
    scd2_table.alias("target")
    .merge(
        staged.alias("source"),
        "target.customer_id = source.customer_id AND target.is_current = true"
    )
    .whenMatchedUpdate(
        condition="target.segment <> source.segment OR target.city <> source.city",
        set={
            "end_date": "current_date()",
            "is_current": "false"
        }
    )
    .execute()
)

print("Step A done: matching current rows expired where segment/city changed.")
scd2_table.toDF().filter("is_current = false").show(5, truncate=False)

Step A done: matching current rows expired where segment/city changed.
+-----------+--------------+-----------+-------------+-------------+--------------+-----------+-------+--------------+----------+----------+
|customer_id|customer_name |segment    |country      |city         |state         |postal_code|region |effective_date|end_date  |is_current|
+-----------+--------------+-----------+-------------+-------------+--------------+-----------+-------+--------------+----------+----------+
|BF-10975   |Barbara Fisher|Corporate  |United States|Charlotte    |North Carolina|28205.0    |South  |2026-08-02    |2026-08-02|false     |
|DW-13480   |Dianna Wilson |Home Office|United States|Lakeville    |Minnesota     |55044.0    |Central|2026-08-02    |2026-08-02|false     |
|PV-18985   |Paul Van Hugh |Home Office|United States|San Francisco|California    |94122.0    |West   |2026-08-02    |2026-08-02|false     |
|RS-19870   |Roy Skaria    |Home Office|United States|Burlington   |Iowa          |

Step B: insert a new current row for both the customers just expired and the brand new customers.

Note: the "Rows inserted" count printed below can look wrong on a second read of this cell, since re-evaluating the lazy DataFrame after the write already changed the table underneath it. The actual insert (30 rows: 20 updated + 10 new) already happened correctly; trust the validation counts in the next cell over this print.

In [19]:
new_versions = staged.join(
    scd2_table.toDF().filter("is_current = false").select("customer_id"),
    on="customer_id", how="inner"
)
brand_new = staged.join(
    scd2_table.toDF().select("customer_id").distinct(),
    on="customer_id", how="left_anti"
)
to_insert = new_versions.unionByName(brand_new)
to_insert.write.format("delta").mode("append").save(scd2_path)

print("Step B done. Rows inserted:", to_insert.count())

Step B done. Rows inserted: 20


## Validate the SCD2 merge

Check total row count, current vs expired counts, and confirm no customer has more than one active (is_current = true) row at a time.

In [20]:
scd2_final = spark.read.format("delta").load(scd2_path)

print("Total rows in SCD2 table (all history):", scd2_final.count())
print("Current rows (is_current = true):", scd2_final.filter("is_current = true").count())
print("Expired rows (is_current = false):", scd2_final.filter("is_current = false").count())

# a customer_id should never have more than one current row at a time
multi_current = scd2_final.filter("is_current = true").groupBy("customer_id").count().filter("count > 1")
print("customer_ids with more than one current row (should be 0):", multi_current.count())

# a customer_id that was updated should have exactly one current + one expired row
history_check = scd2_final.groupBy("customer_id").count().filter("count > 1")
print("customer_ids with more than one row total (i.e. history preserved):", history_check.count())

Total rows in SCD2 table (all history): 815
Current rows (is_current = true): 795
Expired rows (is_current = false): 20
customer_ids with more than one current row (should be 0): 0
customer_ids with more than one row total (i.e. history preserved): 20


## Final output

Compare SCD1 (point in time, no history) against SCD2 (current records) and show one customer's full history as an example of what SCD2 preserves.

In [21]:
print("=== SCD1 table: point-in-time, no history ===")
scd1_df.orderBy("customer_id").show(4, truncate=False)

print("=== SCD2 table: current records only ===")
scd2_final.filter("is_current = true").orderBy("customer_id").show(4, truncate=False)

print("=== SCD2 table: one customer's full history (example) ===")
scd2_final.filter("customer_id = 'DH-13675'").show(truncate=False)

=== SCD1 table: point-in-time, no history ===
+-----------+-------------+--------+-------------+-----------+--------------+-----------+-------+
|customer_id|customer_name|segment |country      |city       |state         |postal_code|region |
+-----------+-------------+--------+-------------+-----------+--------------+-----------+-------+
|AA-10315   |Alex Avila   |Consumer|United States|Minneapolis|Minnesota     |55407.0    |Central|
|AA-10375   |Allen Armold |Consumer|United States|Mesa       |Arizona       |85204.0    |West   |
|AA-10480   |Andrew Allen |Consumer|United States|Concord    |North Carolina|28027.0    |South  |
|AA-10645   |Anna Andreadi|Consumer|United States|Chester    |Pennsylvania  |19013.0    |East   |
+-----------+-------------+--------+-------------+-----------+--------------+-----------+-------+
only showing top 4 rows

=== SCD2 table: current records only ===
+-----------+-------------+--------+-------------+-----------+--------------+-----------+-------+-------

Loaded Superstore customer data into a Delta table (798 raw rows, cleaned to 785 by removing 5 duplicate rows and 8 rows with null segment/postal_code). Simulated an incremental batch of 30 records (20 updates, 10 new customers). Applied an SCD Type 1 merge — updated matched rows in place, inserted unmatched rows, no history kept (785 → 795 rows, 0 duplicate keys). Separately applied an SCD Type 2 merge on a fresh copy of the cleaned data — expired the old version of any changed row (end_date set, is_current = false) and inserted a new current version, preserving full history (785 → 815 rows: 795 current, 20 expired, 0 customers with more than one active row at a time).